# OpenAlex Content Download Workflow

This notebook demonstrates how to use the OpenAlex Content Downloader to bulk download PDFs and TEI XML files.

## Setup

First, install the package if you haven't already:

In [ ]:
# !pip install openalex-content-downloader

In [ ]:
import os
import asyncio

# Set your API key
os.environ["OPENALEX_API_KEY"] = "your-api-key-here"

from openalex_content.api_client import OpenAlexAPIClient
from openalex_content.downloader import DownloadConfig, DownloadOrchestrator
from openalex_content.progress import ProgressTracker
from openalex_content.utils import ContentFormat, StorageType

## Check API Status

First, let's verify our API key works and check our rate limits:

In [ ]:
async def check_status():
    client = OpenAlexAPIClient(api_key=os.environ["OPENALEX_API_KEY"])
    try:
        status = await client.get_status()
        print(f"Rate limit remaining: {status.rate_limit_remaining:,}")
    finally:
        await client.close()

await check_status()

## Explore Available Content

Let's see how many works match our filter before downloading:

In [ ]:
import aiohttp

async def count_works(filter_str: str):
    """Count works matching a filter."""
    api_key = os.environ["OPENALEX_API_KEY"]
    url = "https://api.openalex.org/works"
    params = {
        "filter": f"has_content.pdf:true,{filter_str}",
        "per-page": 1,
        "api_key": api_key,
    }
    
    async with aiohttp.ClientSession() as session:
        async with session.get(url, params=params) as response:
            data = await response.json()
            return data.get("meta", {}).get("count", 0)

# Example: count recent machine learning articles
filter_str = "publication_year:2024,topics.id:T10207"
count = await count_works(filter_str)
print(f"Works matching filter: {count:,}")

## Download Content

Now let's download the content. This example downloads PDFs for recent machine learning articles:

In [ ]:
# Configure the download
config = DownloadConfig(
    api_key=os.environ["OPENALEX_API_KEY"],
    output_path="./ml_papers_2024",
    storage_type=StorageType.LOCAL,
    filter_str="publication_year:2024,topics.id:T10207",
    content_format=ContentFormat.PDF,
    with_metadata=True,  # Save JSON metadata too
    workers=20,  # Moderate concurrency for notebook
)

# Note: In a notebook, we use a simpler progress display
progress = ProgressTracker(
    output_dir=config.output_path,
    quiet=True,  # Use quiet mode in notebooks
)

orchestrator = DownloadOrchestrator(config)

print("Starting download...")
print("Press interrupt (stop button) to pause - progress is saved!")

try:
    await orchestrator.run(progress_tracker=progress)
except KeyboardInterrupt:
    print("\nDownload paused. Run this cell again to resume.")

print(f"\nDownloaded: {progress.stats.total_downloaded} files")
print(f"Failed: {progress.stats.total_failed} files")

## Examine Downloaded Files

Let's look at what we downloaded:

In [ ]:
from pathlib import Path
import json

output_dir = Path("./ml_papers_2024")

# Count files
pdf_files = list(output_dir.rglob("*.pdf"))
json_files = list(output_dir.rglob("*.json"))

print(f"PDF files: {len(pdf_files)}")
print(f"JSON metadata files: {len(json_files)}")

# Look at one metadata file
if json_files:
    with open(json_files[0]) as f:
        metadata = json.load(f)
    print(f"\nExample metadata:")
    print(f"  Title: {metadata.get('title', 'N/A')[:80]}...")
    print(f"  DOI: {metadata.get('doi', 'N/A')}")
    print(f"  Publication date: {metadata.get('publication_date', 'N/A')}")

## Tips

1. **Filtering**: Use OpenAlex filters to narrow down your download. See [filter documentation](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).

2. **Resuming**: If interrupted, just run the download cell again - it will resume from where it left off.

3. **Storage**: For large downloads, consider using S3 storage:
   ```python
   config = DownloadConfig(
       storage_type=StorageType.S3,
       s3_bucket="my-bucket",
       s3_prefix="openalex/",
       ...
   )
   ```

4. **CLI**: For production use, the CLI is easier:
   ```bash
   openalex-content download --api-key KEY --output ./pdfs --filter "publication_year:2024"
   ```